# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kobeyvines/flyrank/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

*Optional deeper sibling to `w04_baseline_score.ipynb` — not the required w04 deliverable. Reuses the five-feature frame from `w03_feature_leakage_check.ipynb` and extends the two required signal checks to three, with distributions up front.*

## 1. Distributions

Looking before deciding. All five features from the w03 frame, plus a note on heavy tails — impressions and clicks-style fields are almost always right-skewed, so a mean alone would mislead.

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_RELATIVE_PATH = Path("data/raw/content_refresh_anonymized.csv")
DATA_PATH = next(
    (base / DATA_RELATIVE_PATH for base in [Path.cwd(), *Path.cwd().parents]
     if (base / DATA_RELATIVE_PATH).is_file()),
    None,
)
if DATA_PATH is None:
    raise FileNotFoundError(
        f"Could not find {DATA_RELATIVE_PATH} from the current directory or its parents. "
        f"Current directory: {Path.cwd()}"
    )
raw = pd.read_csv(DATA_PATH)
raw = raw[(raw["impressions_90d"] > 0) & (raw["content_age_days"] >= 90)].copy()
raw = raw.drop_duplicates(subset="content_id")

FEATURES = ["impressions_90d", "avg_position", "ctr", "days_since_last_update", "word_count"]

desc = raw[FEATURES].describe(percentiles=[0.5, 0.9, 0.99]).T
print(desc.to_string())

# Heavy-tail check: compare mean vs. 90th/99th percentile. A mean well below the 99th percentile
# means a handful of high-volume pages dominate -- any threshold picked later (like MIN_IMPRESSIONS
# in the baseline notebook) should be sanity-checked against this shape, not against the mean.
for f in ["impressions_90d", "word_count"]:
    print(f"\n{f}: mean={raw[f].mean():.1f}, median={raw[f].median():.1f}, "
          f"p90={raw[f].quantile(0.9):.1f}, p99={raw[f].quantile(0.99):.1f} "
          f"-- {'heavy right tail' if raw[f].mean() < raw[f].quantile(0.5) * 3 else 'very heavy right tail'}")

print(f"\navg_position == 0 or missing (no reported ranking): {(raw['avg_position'] <= 0).sum():,} rows")

                          count         mean           std  min      50%       90%        99%       max
impressions_90d         30000.0  5200.366300  16838.019547  1.0   731.00  12136.40  73505.830  517715.0
avg_position            30000.0    16.342380     15.216790  0.0    10.80     36.80     69.901     245.0
ctr                     30000.0     0.510733      3.279162  0.0     0.07      0.65      8.330     100.0
days_since_last_update  30000.0    46.098300     42.078709  1.0    20.00    104.00    106.000     373.0
word_count              22301.0  3107.760325   1452.382598  8.0  2877.00   5327.00   7292.000    9546.0

impressions_90d: mean=5200.4, median=731.0, p90=12136.4, p99=73505.8 -- very heavy right tail

word_count: mean=3107.8, median=2877.0, p90=5327.0, p99=7292.0 -- heavy right tail

avg_position == 0 or missing (no reported ranking): 1,205 rows


## 2. Signal test #1 / #2 / #3 (verdict each)

Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE. Tests 1 and 2 are the same ones from `w04_baseline_score.ipynb` (CTR-by-position-tier, staleness-by-impressions); test 3 is new here — word count vs. position, checking the loose industry assumption that longer content ranks better.

**Test 1 — CTR vs. position tier** (flag-linked: `low_ctr_visible_page`).
**Test 2 — Staleness vs. impressions** (flag-linked: `stale_visible_page`).
**Test 3 — Word count vs. position tier** (not flag-linked — an independent check of a common content-team assumption, useful as a contrast case against the two flag-linked tests above).

In [4]:
has_position = raw["avg_position"] > 0
raw["position_tier"] = pd.Series(index=raw.index, dtype="object")
raw.loc[has_position, "position_tier"] = pd.qcut(
    raw.loc[has_position, "avg_position"], 5, labels=[1, 2, 3, 4, 5]
)

# --- Test 1: CTR vs. position tier ---
t1 = raw.loc[has_position].groupby("position_tier", observed=True).agg(
    n=("ctr", "size"), median_ctr=("ctr", "median")
).reset_index()
print("Test 1 -- CTR by position tier")
print(t1.to_string(index=False))
verdict1 = "CONFIRMED" if t1.set_index("position_tier")["median_ctr"].is_monotonic_decreasing else "MIXED"
print(f"Verdict 1: {verdict1} (n={len(raw.loc[has_position]):,})\n")

# --- Test 2: Staleness vs. impressions ---
raw["staleness_bucket"] = pd.cut(
    raw["days_since_last_update"], bins=[-1, 90, 180, np.inf], labels=["<90d", "90-180d", "180d+"]
)
t2 = raw.groupby("staleness_bucket", observed=True).agg(
    n=("impressions_90d", "size"), median_impressions=("impressions_90d", "median")
).reset_index()
print("Test 2 -- Impressions by staleness bucket")
print(t2.to_string(index=False))
stale_med = t2.loc[t2["staleness_bucket"] == "180d+", "median_impressions"].iloc[0]
fresh_med = t2.loc[t2["staleness_bucket"] == "<90d", "median_impressions"].iloc[0]
verdict2 = "CONFIRMED" if stale_med >= 0.5 * fresh_med else "MIXED"
print(f"Verdict 2: {verdict2} (n={len(raw):,})\n")

# --- Test 3: Word count vs. position tier ---
t3 = raw.loc[has_position].groupby("position_tier", observed=True).agg(
    n=("word_count", "size"), median_word_count=("word_count", "median")
).reset_index()
print("Test 3 -- Word count by position tier")
print(t3.to_string(index=False))
wc_by_tier = t3.set_index("position_tier")["median_word_count"]
# "Longer ranks better" would show DEcreasing word count as tier worsens (tier 1 = best position).
if wc_by_tier.is_monotonic_decreasing:
    verdict3 = "CONFIRMED"
elif wc_by_tier.is_monotonic_increasing:
    verdict3 = "OPPOSITE"
else:
    verdict3 = "MIXED"
print(f"Verdict 3: {verdict3} (n={len(raw.loc[has_position]):,})")

# Read all three tables above before accepting these verdicts -- the monotonic check is a first
# pass, not a substitute for looking at the actual numbers, especially for Test 3 where a FALSE
# or OPPOSITE result is a genuinely useful, publishable finding (it would mean length alone isn't
# doing the work people assume it does).

Test 1 -- CTR by position tier
 position_tier    n  median_ctr
             1 5942       0.180
             2 5595       0.130
             3 5802       0.110
             4 5708       0.085
             5 5748       0.000
Verdict 1: CONFIRMED (n=28,795)

Test 2 -- Impressions by staleness bucket
staleness_bucket     n  median_impressions
            <90d 20655               472.0
         90-180d  9171              1692.0
           180d+   174                15.5
Verdict 2: MIXED (n=30,000)

Test 3 -- Word count by position tier
 position_tier    n  median_word_count
             1 5942             2740.0
             2 5595             2842.0
             3 5802             2880.0
             4 5708             3126.0
             5 5748             3016.0
Verdict 3: MIXED (n=28,795)


## 3. The flag-linked test

Of the three tests above, **Test 1 (CTR vs. position tier)** is the one to hold up as the headline flag-linked result — it maps directly onto FlyRank's `low_ctr_visible_page` flag, and its CONFIRMED/MIXED verdict directly tells you whether that flag's core assumption (low CTR at a given position is a real, distinguishable pattern) holds in this data. Test 2 is the secondary flag-linked check, tied to `stale_visible_page`.

In [5]:
print("Flag-linked verdict summary:")
print(f"  low_ctr_visible_page  <- Test 1 (CTR vs. position tier): {verdict1}")
print(f"  stale_visible_page    <- Test 2 (staleness vs. impressions): {verdict2}")
print(f"  (no flag; contrast case) Test 3 (word count vs. position tier): {verdict3}")

# If either flag-linked verdict comes back FALSE or OPPOSITE, that's not a failure of this notebook --
# it's evidence FlyRank's existing rule rests on an assumption this data doesn't support, and that
# is exactly the kind of honest finding the lane guide (section 13) asks for.

Flag-linked verdict summary:
  low_ctr_visible_page  <- Test 1 (CTR vs. position tier): CONFIRMED
  stale_visible_page    <- Test 2 (staleness vs. impressions): MIXED
  (no flag; contrast case) Test 3 (word count vs. position tier): MIXED


## 4. What this means in practice

*(Fill in once the actual verdicts above are read by hand — this is a placeholder shape, not a substitute for writing it after seeing the real numbers.)*

If Test 1 comes back CONFIRMED: a content team can trust position-tier-adjusted CTR gaps as a real prioritization signal for metadata review — this is what the baseline rule in `w04_baseline_score.ipynb` depends on. If Test 2 comes back CONFIRMED: staleness plus a volume floor is a reasonable trigger for a refresh queue, but staleness alone, without the volume check, would waste review time on pages nobody sees. If Test 3 comes back anything other than CONFIRMED: word count should not be used as a standalone ranking-signal proxy in this dataset, whatever the industry default assumption says.

In [6]:
# Optional: a short automated line if you want the notebook to state its own conclusion.
print("Practical takeaway (edit this string by hand once verdicts are confirmed):")
print(f"  Test 1 ({verdict1}) -> position-tier-adjusted CTR is", 
      "a trustworthy metadata-review trigger." if verdict1 == "CONFIRMED" else "NOT a clean trigger on its own -- revisit before using it in the baseline rule.")

Practical takeaway (edit this string by hand once verdicts are confirmed):
  Test 1 (CONFIRMED) -> position-tier-adjusted CTR is a trustworthy metadata-review trigger.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.